# Data Loading Tutorial

This tutorial covers how to load EEG data from various formats supported by NeuRodent.

## Overview

NeuRodent supports multiple data formats commonly used in rodent EEG research:

1. **SpikeInterface recordings** (`mode="si"`) - Any format supported by SpikeInterface (EDF, Intan, Open Ephys, NWB, etc.)
2. **MNE objects** (`mode="mne"`) - Any format supported by MNE-Python (FIF, EDF, BDF, etc.)
3. **Pre-created recordings** (`mode=None`) - Pass an existing `si.BaseRecording` directly
4. **Custom formats** - Via a custom `extract_func` callable or module path string

The `LongRecordingOrganizer` (LRO) class handles loading and organizing recordings from these formats.
The `AnimalOrganizer` class manages multiple sessions for a single animal using glob-style patterns.

## Setup

In [ ]:
import sys
from pathlib import Path
import logging

from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt

from neurodent import core
from neurodent import constants

import mne
import spikeinterface.core as si
import spikeinterface.extractors as se


# Set up logging
logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(message)s", 
    level=logging.INFO
)
logger = logging.getLogger()

## 1. Loading EDF Files via SpikeInterface

The most common way to load data is via `mode="si"` with an `extract_func` that tells NeuRodent which SpikeInterface reader to use.

In [ ]:
# Example: Loading an EDF file via SpikeInterface
# Using included test data
data_path = Path("../../notebooks/tests/test-data/A10 KO 12_13_2023/A10_si.edf")

# Create LongRecordingOrganizer
# mode options: 'si' (SpikeInterface), 'mne' (MNE-Python), None (pre-created recording)
lro_si = core.LongRecordingOrganizer(
    item=data_path,
    mode="si",
    extract_func="read_edf",  # SpikeInterface extractor name
)

print(f"Sampling frequency: {lro_si.meta.f_s}")
print(f"Number of channels: {lro_si.meta.n_channels}")
print(f"Channel names: {lro_si.meta.channel_names}")

In [ ]:
# Access the underlying SpikeInterface recording
recording = lro_si.LongRecording

# You can also convert to MNE for visualization
# lro_si_mne = lro_si.convert_to_mne()

print(f"Recording type: {type(recording).__name__}")
print(f"Duration: {recording.get_total_duration():.1f} seconds")

### Custom Extract Functions

You can pass a custom `extract_func` to handle non-standard formats. The function should accept a file path and return a `si.BaseRecording` (for `mode="si"`) or `mne.io.Raw` (for `mode="mne"`).

You can also specify `extract_func` as a module path string (e.g., `"my_module.py:read_custom"`).

In [ ]:
# Example: Using a custom extract_func as a callable
def my_custom_reader(file_path):
    """Custom reader that returns a SpikeInterface recording."""
    import spikeinterface.extractors as se
    return se.read_edf(file_path)

lro_custom_func = core.LongRecordingOrganizer(
    item="/path/to/recording.edf",
    mode="si",
    extract_func=my_custom_reader,
)

# Example: Using a module path string as extract_func
# lro_custom_str = core.LongRecordingOrganizer(
#     item="/path/to/recording.bin",
#     mode="si",
#     extract_func="my_readers.py:read_bin_csv_pair",
# )

## 2. Loading SpikeInterface Recordings

SpikeInterface is a popular Python library for extracellular electrophysiology data. NeuRodent can directly use SpikeInterface recordings:

In [ ]:
# need pyedflib

# Example: Loading an EDF file with manual datetime
si_data_path = Path("../../notebooks/tests/test-data/A10 KO 12_13_2023/A10_si.edf")

lro_si2 = core.LongRecordingOrganizer(
    item=si_data_path,
    mode="si",
    manual_datetimes=datetime(2023, 12, 12),
    extract_func=se.read_edf,
)

print(f"Sampling frequency: {lro_si2.meta.f_s}")
print(f"Number of channels: {lro_si2.meta.n_channels}")

## 3. Loading MNE Objects

MNE-Python is a widely-used library for MEG and EEG analysis. NeuRodent can work with MNE Raw objects:

In [ ]:
import mne
from datetime import datetime
from pathlib import Path

data_path = Path("../../notebooks/tests/test-data/A10 KO 12_13_2023/A10_mne.fif")

# Create LongRecordingOrganizer with MNE mode
lro_mne = core.LongRecordingOrganizer(
    item=data_path,
    manual_datetimes=datetime(2023, 12, 13),
    mode="mne",
    extract_func=mne.io.read_raw_fif,
)

print(f"Sampling frequency: {lro_mne.meta.f_s}")
print(f"Number of channels: {lro_mne.meta.n_channels}")

## 4. Inspecting Loaded Data

Once data is loaded, you can inspect its properties via the `meta` attribute (a `RecordingMetadata` object):

In [ ]:
# Get basic properties using metadata from lro object

metadata = lro_si.meta

print(f"Recording metadata: {metadata}")

print(f"Sampling frequency: {metadata.f_s} Hz")
print(f"Number of channels: {metadata.n_channels}")
print(f"Channel names: {metadata.channel_names}")
print(f"Units: {metadata.V_units}")


print(f"Duration: {lro_si.file_durations} seconds")

## 5. Loading Other Formats

Neurodent supports other data formats as well, such as Neuroscope/Neuralyns, Ephys, NWB files, etc.
Although the example datasets are not provided, sections below serve as a starting point for loading these formats.

### Neuroscope/Neuralynx

For `.dat` or `.eeg` files:

In [ ]:
# Load using SpikeInterface extractors, then wrap in LRO
neuroscope_path = Path("/path/to/neuroscope/data.dat")
recording_neuroscope = se.read_neuroscope(neuroscope_path)

lro_neuroscope = core.LongRecordingOrganizer(
    item=None,
    mode=None,
    recording=recording_neuroscope,
)

### Open Ephys

For Open Ephys `.continuous` files:

In [ ]:
# Load using SpikeInterface extractors, then wrap in LRO
openephys_path = Path("/path/to/openephys/folder")
recording_openephys = se.read_openephys(openephys_path)

lro_openephys = core.LongRecordingOrganizer(
    item=None,
    mode=None,
    recording=recording_openephys,
)

### NWB Files

Neurodata Without Borders (NWB) is a standardized format for neurophysiology data:

In [ ]:
# Example: Loading NWB files
nwb_path = Path("/path/to/nwb/file.nwb")

# First, load with SpikeInterface's NWB extractor
import spikeinterface.extractors as se

recording_nwb = se.read_nwb(nwb_path)

# Then wrap in LongRecordingOrganizer
lro_nwb = core.LongRecordingOrganizer(
    item=None,
    mode=None,
    recording=recording_nwb,
)

print(f"Loaded NWB data")
print(f"Sampling frequency: {lro_nwb.meta.f_s}")
print(f"Number of channels: {lro_nwb.meta.n_channels}")

## 6. Working with Multiple Recordings

NeuRodent can handle multiple recordings from the same animal (e.g., different sessions or days):

In [ ]:
# Example: Loading multiple files as a concatenated recording
# Pass a list of file paths to concatenate them in order
file_list = [
    "/path/to/session1/recording.edf",
    "/path/to/session2/recording.edf",
]

lro_multi = core.LongRecordingOrganizer(
    item=file_list,
    mode="si",
    extract_func="read_edf",
)

print(f"File durations: {lro_multi.file_durations}")
print(f"Total duration: {sum(lro_multi.file_durations):.1f} seconds")

## 7. Advanced: Custom Data Loading

For custom formats, you can create SpikeInterface Recording objects and pass them to `LongRecordingOrganizer`:

In [ ]:
import spikeinterface.core as si

# Example: Create a recording from numpy array
# (useful for custom formats or testing)
num_channels = 16
sampling_frequency = 1000  # Hz
duration = 60  # seconds
num_samples = int(sampling_frequency * duration)

# Generate random data (replace with your actual data)
data = np.random.randn(num_channels, num_samples)

# Create SpikeInterface recording
recording_custom = si.NumpyRecording(
    traces_list=[data],
    sampling_frequency=sampling_frequency,
)

# Set channel IDs
channel_ids = [f"CH{i:02d}" for i in range(num_channels)]
recording_custom = recording_custom.rename_channels(
    new_channel_ids=channel_ids
)

# Use with LongRecordingOrganizer (mode=None for pre-created recordings)
lro_custom = core.LongRecordingOrganizer(
    item=None,
    mode=None,
    recording=recording_custom,
)

print("Custom recording created successfully!")
print(f"Sampling frequency: {lro_custom.meta.f_s}")
print(f"Number of channels: {lro_custom.meta.n_channels}")

## Summary

In this tutorial, you learned:

1. How to load data using `LongRecordingOrganizer` with `mode="si"` and `mode="mne"`
2. How to use custom `extract_func` callables or module path strings
3. How to wrap pre-created `si.BaseRecording` objects with `mode=None`
4. How to inspect loaded data properties via the `meta` attribute
5. How to concatenate multiple files and create custom recordings

## Next Steps

- **[Basic Usage Tutorial](basic_usage.ipynb)**: Complete workflow from loading to visualization
- **[Windowed Analysis Tutorial](../tutorials/windowed_analysis.ipynb)**: Extract features from loaded data
- **[Spike Analysis Tutorial](../tutorials/spike_analysis.ipynb)**: Work with spike-sorted data